<div style="background: linear-gradient(135deg, #e8f4ff 0%, #fffaf0 100%); border: 1px solid #b7d8f5; border-left: 9px solid #1f6fb2; border-radius: 14px; padding: 28px; font-family: Georgia, 'Times New Roman', serif; color: #17324d;">
<h1 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin: 0 0 10px 0;">Transformers desde cero con PyTorch</h1>
<p style="font-size: 1.15em; margin: 0 0 14px 0;"><strong>Implementación académica y pedagógica de un Transformer codificador-decodificador.</strong></p>
<p style="margin: 0; line-height: 1.6;">Este cuaderno desarrolla los mecanismos centrales de un Transformer: codificación posicional, atención escalada, atención multi-cabeza, conexiones residuales, normalización de capas, redes de avance por posición y atención cruzada.</p>
</div>

<br>

<div style="background-color: #fffaf0; border: 1px solid #eadfca; border-radius: 10px; padding: 16px 20px; font-family: Georgia, 'Times New Roman', serif; color: #4a3b2a;">
<strong>Propósito.</strong> Convertir conceptos arquitectónicos en código legible, modificable e inspeccionable. Esta implementación es educativa; para producción, considera las versiones optimizadas de PyTorch o Hugging Face.
</div>


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">1. Preparación del entorno</h2>

Se importan los módulos necesarios de PyTorch y se selecciona automáticamente una GPU cuando está disponible. La semilla fija permite reproducir los tensores sintéticos del ejemplo.

<ul>
<li><code>math</code> aporta la raíz cuadrada usada en la atención escalada.</li>
<li><code>torch</code> proporciona tensores y cálculo numérico acelerado.</li>
<li><code>torch.nn</code> contiene capas y módulos entrenables.</li>
<li><code>torch.nn.functional</code> incluye funciones como <code>softmax</code>.</li>
</ul>
</div>


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo seleccionado: {device}")


<div style="background-color: #fffaf0; border-left: 7px solid #c58b39; border-radius: 8px; padding: 16px 20px; font-family: Georgia, 'Times New Roman', serif; color: #4a3b2a; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #835b1c; margin-top: 0;">2. Hiperparámetros</h2>

Los valores son moderados para permitir ejecución en Jupyter o Google Colab. En aplicaciones reales, la elección de estos parámetros debe equilibrar calidad, memoria, latencia y coste computacional.

<ul>
<li><code>D_MODEL</code>: dimensión interna de cada token.</li>
<li><code>NUM_HEADS</code>: perspectivas paralelas de atención.</li>
<li><code>NUM_LAYERS</code>: profundidad del codificador y decodificador.</li>
<li><code>D_FF</code>: dimensión interna de la red de avance.</li>
<li><code>PAD_IDX</code>: identificador para posiciones de relleno.</li>
</ul>
</div>


In [ ]:
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 4
D_FF = 1024
DROPOUT = 0.1
MAX_SEQ_LENGTH = 128
SRC_VOCAB_SIZE = 10_000
TGT_VOCAB_SIZE = 10_000
PAD_IDX = 0

assert D_MODEL % NUM_HEADS == 0, "D_MODEL debe ser divisible por NUM_HEADS."
print(f"Dimensión por cabeza: {D_MODEL // NUM_HEADS}")


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">3. Codificación posicional</h2>

La autoatención no conoce por sí sola el orden de los tokens. La codificación posicional añade una señal dependiente de la posición a cada embedding antes de que entre en las capas del Transformer.

<strong>Guía de lectura:</strong>
<ul>
<li><code>positions</code> crea índices de posición.</li>
<li><code>frequencies</code> construye escalas de frecuencia geométricas.</li>
<li>Las funciones seno y coseno producen patrones distinguibles por posición.</li>
<li><code>register_buffer</code> conserva la matriz sin hacerla entrenable.</li>
</ul>
</div>


In [ ]:
class PositionalEncoding(nn.Module):
    """Añade información posicional sinusoidal a tensores (lote, longitud, d_model)."""

    def __init__(self, d_model, max_seq_length, dropout=0.1):
        super().__init__()
        positions = torch.arange(max_seq_length).unsqueeze(1).float()
        frequencies = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10_000.0) / d_model)
        )

        pe = torch.zeros(max_seq_length, d_model)
        pe[:, 0::2] = torch.sin(positions * frequencies)
        pe[:, 1::2] = torch.cos(positions * frequencies)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">4. Atención de producto escalar escalada</h2>

La atención estima qué tokens son relevantes para representar cada posición. Las consultas expresan qué información busca una posición; las claves describen lo disponible; y los valores aportan el contenido que se combinará.

<ul>
<li>Se calculan similitudes entre consultas y claves.</li>
<li>Se escalan los puntajes para favorecer estabilidad numérica.</li>
<li>Las máscaras excluyen relleno o posiciones futuras.</li>
<li><code>softmax</code> convierte puntajes en pesos de atención.</li>
<li>Los valores se combinan usando dichos pesos.</li>
</ul>
</div>


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    """Calcula atención escalada para q, k, v: (lote, cabezas, longitud, dimensión)."""
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    attention_weights = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    output = attention_weights @ v
    return output, attention_weights


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">5. Atención multi-cabeza</h2>

La atención multi-cabeza realiza varias operaciones de atención sobre proyecciones diferentes de una misma entrada. Sus resultados se concatenan y se integran en una única representación.

<ul>
<li><code>w_q</code>, <code>w_k</code> y <code>w_v</code> producen consultas, claves y valores.</li>
<li><code>split_heads</code> divide la dimensión del modelo en cabezas paralelas.</li>
<li><code>combine_heads</code> reconstruye la representación final.</li>
<li><code>w_o</code> proyecta la salida al espacio del modelo.</li>
</ul>
</div>


In [ ]:
class MultiHeadAttention(nn.Module):
    """Implementación pedagógica de atención multi-cabeza."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads."
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.shape
        x = x.view(batch_size, seq_length, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_length, self.num_heads * self.d_k)

    def forward(self, query, key, value, mask=None, return_attention=False):
        q = self.split_heads(self.w_q(query))
        k = self.split_heads(self.w_k(key))
        v = self.split_heads(self.w_v(value))
        attended, weights = scaled_dot_product_attention(q, k, v, mask, self.dropout)
        output = self.w_o(self.combine_heads(attended))
        return (output, weights) if return_attention else output


<div style="background-color: #fffaf0; border-left: 7px solid #c58b39; border-radius: 8px; padding: 16px 20px; font-family: Georgia, 'Times New Roman', serif; color: #4a3b2a; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #835b1c; margin-top: 0;">6. Red de avance por posición</h2>

Después de que la atención combina contexto entre tokens, esta red no lineal transforma cada posición de manera independiente. La expansión interna aumenta la capacidad expresiva y la proyección final recupera la dimensión necesaria para las conexiones residuales.
</div>


In [ ]:
class FeedForward(nn.Module):
    """Transformación no lineal aplicada a cada posición de la secuencia."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">7. Capa del codificador</h2>

Cada capa del codificador alterna autoatención y red de avance. Se usa pre-normalización: cada subcapa recibe una entrada normalizada y su salida se suma a la representación anterior mediante una conexión residual.
</div>


In [ ]:
class EncoderLayer(nn.Module):
    """Autoatención + red de avance con pre-normalización y conexiones residuales."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        normalized_x = self.norm1(x)
        x = x + self.dropout(self.self_attention(normalized_x, normalized_x, normalized_x, src_mask))
        normalized_x = self.norm2(x)
        return x + self.dropout(self.feed_forward(normalized_x))


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">8. Capa del decodificador</h2>

El decodificador incorpora tres operaciones: autoatención causal, atención cruzada y red de avance. La máscara causal impide que cada posición consulte tokens futuros; la atención cruzada permite consultar la representación de la secuencia fuente creada por el codificador.
</div>


In [ ]:
class DecoderLayer(nn.Module):
    """Autoatención causal + atención cruzada + red de avance."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.masked_self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        normalized_x = self.norm1(x)
        x = x + self.dropout(self.masked_self_attention(normalized_x, normalized_x, normalized_x, tgt_mask))
        normalized_x = self.norm2(x)
        x = x + self.dropout(self.cross_attention(normalized_x, encoder_output, encoder_output, src_mask))
        normalized_x = self.norm3(x)
        return x + self.dropout(self.feed_forward(normalized_x))


<div style="background-color: #e8f4ff; border: 1px solid #b7d8f5; border-left: 7px solid #277da1; border-radius: 10px; padding: 18px 22px; font-family: Georgia, 'Times New Roman', serif; color: #17324d; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">9. Transformer codificador-decodificador completo</h2>

El modelo integra embeddings, posiciones, máscaras, capas del codificador, capas del decodificador y una proyección final sobre el vocabulario objetivo.

<ul>
<li>La máscara de relleno evita atender posiciones sin contenido.</li>
<li>La máscara causal impide que el decodificador vea el futuro.</li>
<li>La proyección final produce logits, que se entregan directamente a la función de pérdida.</li>
</ul>
</div>


In [ ]:
class Transformer(nn.Module):
    """Transformer codificador-decodificador para fines educativos."""

    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=D_MODEL, num_heads=NUM_HEADS,
                 num_layers=NUM_LAYERS, d_ff=D_FF, max_seq_length=MAX_SEQ_LENGTH, dropout=DROPOUT):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length, dropout)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model

    @staticmethod
    def create_padding_mask(tokens, pad_idx=PAD_IDX):
        return (tokens != pad_idx).unsqueeze(1).unsqueeze(2)

    @staticmethod
    def create_causal_mask(sequence_length, device):
        return torch.tril(torch.ones(sequence_length, sequence_length, device=device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)

    def forward(self, src, tgt_input, pad_idx=PAD_IDX):
        src_mask = self.create_padding_mask(src, pad_idx)
        tgt_mask = self.create_padding_mask(tgt_input, pad_idx) & self.create_causal_mask(tgt_input.size(1), tgt_input.device)
        encoder_output = self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        decoder_output = self.positional_encoding(self.tgt_embedding(tgt_input) * math.sqrt(self.d_model))

        for layer in self.encoder_layers:
            encoder_output = layer(encoder_output, src_mask)
        for layer in self.decoder_layers:
            decoder_output = layer(decoder_output, encoder_output, tgt_mask, src_mask)

        return self.output_projection(decoder_output)


<div style="background-color: #fffaf0; border-left: 7px solid #c58b39; border-radius: 8px; padding: 16px 20px; font-family: Georgia, 'Times New Roman', serif; color: #4a3b2a; line-height: 1.65;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #835b1c; margin-top: 0;">10. Prueba de ejecución y un paso de entrenamiento</h2>

Se crean datos sintéticos para verificar la coherencia de las dimensiones y calcular una pérdida. En un proyecto real, estos identificadores de token provendrían de un tokenizador y de un conjunto de datos de pares entrada-salida.

<strong>Teacher forcing:</strong> el decodificador recibe la secuencia objetivo desplazada y aprende a predecir el token siguiente.
</div>


In [ ]:
model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE).to(device)

batch_size, src_length, tgt_length = 8, 20, 18
src = torch.randint(1, SRC_VOCAB_SIZE, (batch_size, src_length), device=device)
tgt = torch.randint(1, TGT_VOCAB_SIZE, (batch_size, tgt_length), device=device)

tgt_input = tgt[:, :-1]
tgt_expected = tgt[:, 1:]

logits = model(src, tgt_input)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

model.train()
optimizer.zero_grad()
loss = criterion(logits.reshape(-1, TGT_VOCAB_SIZE), tgt_expected.reshape(-1))
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print(f"Forma de logits: {tuple(logits.shape)}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")
print(f"Pérdida después de un paso: {loss.item():.4f}")


<div style="background: linear-gradient(135deg, #e8f4ff 0%, #fffaf0 100%); border: 1px solid #b7d8f5; border-left: 9px solid #1f6fb2; border-radius: 14px; padding: 24px; font-family: Georgia, 'Times New Roman', serif; color: #17324d;">
<h2 style="font-family: Georgia, 'Times New Roman', serif; color: #0c4a6e; margin-top: 0;">Próximos pasos</h2>

<ul>
<li>Sustituir los datos sintéticos por un corpus de traducción, resumen o reformulación.</li>
<li>Incorporar tokenización real, tokens de inicio, fin y relleno.</li>
<li>Añadir ciclos de validación y métricas como perplejidad.</li>
<li>Implementar decodificación voraz, búsqueda en haz o muestreo.</li>
<li>Comparar con <code>torch.nn.Transformer</code> y Hugging Face Transformers.</li>
<li>Analizar latencia, memoria y calidad en secuencias más largas.</li>
</ul>

<strong>Idea central:</strong> el valor de este cuaderno no consiste en reproducir un modelo de frontera, sino en hacer visibles y modificables los mecanismos que sustentan los Transformers modernos.
</div>
